<a href="https://colab.research.google.com/github/HiroMiyagi1/Contactos/blob/main/trabajo_2_programacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%writefile models.py
from dataclasses import dataclass

@dataclass
class Contacto:
    id: int
    nombre: str
    telefono: str

    def __str__(self) -> str:
        return f"Contacto {self.id} ({self.nombre})"

Overwriting models.py


In [ ]:
%%writefile interfaces.py
from abc import ABC, abstractmethod
from typing import List
from models import Contacto

class IContactoRepository(ABC):
    @abstractmethod
    def save(self, nombre: str, telefono: str) -> Contacto: pass
    @abstractmethod
    def get_all(self) -> List[Contacto]: pass
    @abstractmethod
    def update_telefono_por_nombre(self, nombre: str, nuevo_telefono: str) -> bool: pass
    @abstractmethod
    def delete_por_nombre(self, nombre: str) -> bool: pass

Overwriting interfaces.py


In [ ]:
%%writefile repositories.py
from typing import List, Dict
from models import Contacto
from interfaces import IContactoRepository

class InMemoryContactoRepository(IContactoRepository):
    def __init__(self):
        self._contactos: Dict[int, Contacto] = {}
        self._next_id: int = 1

    def save(self, nombre: str, telefono: str) -> Contacto:
        c = Contacto(id=self._next_id, nombre=nombre, telefono=telefono)
        self._contactos[c.id] = c
        self._next_id += 1
        return c

    def get_all(self) -> List[Contacto]:
        return list(self._contactos.values())

    def update_telefono_por_nombre(self, nombre: str, nuevo_telefono: str) -> bool:
        for contacto in self._contactos.values():
            if contacto.nombre.lower() == nombre.lower():
                contacto.telefono = nuevo_telefono
                return True
        return False

    def delete_por_nombre(self, nombre: str) -> bool:
        for id_c, contacto in list(self._contactos.items()):
            if contacto.nombre.lower() == nombre.lower():
                del self._contactos[id_c]
                return True
        return False

Overwriting repositories.py


In [ ]:
%%writefile services.py
from typing import List
from models import Contacto
from interfaces import IContactoRepository

class ContactoService:
    def __init__(self, repo: IContactoRepository):
        self._repo = repo

    def crear(self, nombre: str, telefono: str) -> Contacto:
        if not nombre.strip():
            raise ValueError("El nombre es obligatorio")
        return self._repo.save(nombre, telefono)

    def listar(self) -> List[Contacto]:
        return self._repo.get_all()

    def actualizar_por_nombre(self, nombre: str, nuevo_tel: str) -> bool:
        if not nombre.strip():
            raise ValueError("El nombre no puede estar vacio")
        return self._repo.update_telefono_por_nombre(nombre, nuevo_tel)

    def eliminar_por_nombre(self, nombre: str) -> bool:
        if not nombre.strip():
            raise ValueError("El nombre a eliminar no puede estar vacio")
        return self._repo.delete_por_nombre(nombre)

Overwriting services.py


In [ ]:
%%writefile main.py
from repositories import InMemoryContactoRepository
from services import ContactoService

def main():
    repo = InMemoryContactoRepository()
    service = ContactoService(repo)

    while True:
        print("\n### REGISTRO DE CONTACTOS ###")
        print("1. Agregar contacto")
        print("2. Lista de contactos")
        print("3. Actualizar telefono")
        print("4. Eliminar contacto")
        print("5. Salir")

        opcion = input("Selecciona una opcion (1-5): ").strip()

        if opcion == "1":
            nombre = input("Igresa el nombre: ").strip()
            telefono = input("Ingresa el telefono: ").strip()
            try:
                c = service.crear(nombre, telefono)
                print(f"\nContacto Guardado!:D {c}")
            except ValueError as e:
                print(f"\nError: {e}")

        elif opcion == "2":
            contactos = service.listar()
            print("\n### LISTA DE CONTACTOS ###")
            if not contactos:
                print("No hay contactos guardados :c")
            else:
                for c in contactos:
                    print(f"ID: {c.id} | Nombre: {c.nombre} | Telefono: {c.telefono}")

        elif opcion == "3":
            contactos = service.listar()
            print("\n### LISTA DE CONTACTOS ###")
            if not contactos:
                print("No hay contactos guardados para actualizar :c")
            else:
                for c in contactos:
                    print(f"ID: {c.id} | Nombre: {c.nombre} | Telefono: {c.telefono}")

                print("-" * 30)
                nombre_a_actualizar = input("Ingresa el nombre del contacto a actualizar: ").strip()
                tel = input("Igresa el nuevo telefono: ").strip()

                try:
                    if service.actualizar_por_nombre(nombre_a_actualizar, tel):
                        print(f"\nTelefono de '{nombre_a_actualizar}' actualizado con exito.")
                    else:
                        print(f"\nNo se encontro ningun contacto con ese nombre '{nombre_a_actualizar}'.")
                except ValueError as e:
                    print(f"\nError: {e}")

        elif opcion == "4":
            contactos = service.listar()
            print("\n### LISTA DE CONTACTOS ###")
            if not contactos:
                print("No hay contactos guardado para eliminar :c")
            else:
                for c in contactos:
                    print(f"ID: {c.id} | Nombre: {c.nombre} | Telefono: {c.telefono}")

                print("-" * 30)
                nombre_a_eliminar = input("Ingresa el nombre del contacto a eliminar: ").strip()
                try:
                    if service.eliminar_por_nombre(nombre_a_eliminar):
                        print(f"\nContacto '{nombre_a_eliminar}' eliminado con exito.")
                    else:
                        print(f"\nNo se encontro ningun contacto con ese nombre '{nombre_a_eliminar}'.")
                except ValueError as e:
                    print(f"\nError: {e}")

        elif opcion == "5":
            print("\nSaliendo del programa...")
            break

if __name__ == "__main__":
    main()

Overwriting main.py


In [ ]:
!python main.py


### REGISTRO DE CONTACTOS ###
1. Agregar contacto
2. Lista de contactos
3. Actualizar telefono
4. Eliminar contacto
5. Salir
Selecciona una opcion (1-5): 